In [0]:
from pyspark.sql.functions import (
    col,
    from_json,
    get_json_object,
    lit,
    sha2,
    concat_ws,
    current_timestamp,
    to_date,
    when,
    trim,
    lower,
    row_number
)

from pyspark.sql.window import Window

print("Silver notebook started")

In [0]:
BRONZE_TABLE = "workspace.fhir.bronze_patient"
SILVER_TABLE = "workspace.fhir.silver_patient"

print("Bronze:", BRONZE_TABLE)
print("Silver:", SILVER_TABLE)

In [0]:
bronze_patient_df = spark.table(BRONZE_TABLE)

display(bronze_patient_df)

In [0]:
print(
    "Bronze Patient count:",
    bronze_patient_df.count()
)

In [0]:
display(
    bronze_patient_df
    .select(
        "resource_id",
        "raw_json"
    )
    .limit(5)
)

In [0]:
silver_patient_df = (
    bronze_patient_df
    .select(
        col("resource_id").alias("patient_id"),

        get_json_object(
            col("raw_json"),
            "$.gender"
        ).alias("gender"),

        get_json_object(
            col("raw_json"),
            "$.birthDate"
        ).alias("birth_date"),

        get_json_object(
            col("raw_json"),
            "$.active"
        ).alias("active"),

        col("source_file"),
        col("ingestion_timestamp"),
        col("raw_json")
    )
)

display(silver_patient_df)

In [0]:
silver_patient_df.printSchema()

In [0]:
silver_patient_df = (
    silver_patient_df
    .withColumn(
        "last_name",
        get_json_object(
            col("raw_json"),
            "$.name[0].family"
        )
    )
    .withColumn(
        "first_name",
        get_json_object(
            col("raw_json"),
            "$.name[0].given[0]"
        )
    )
)

display(
    silver_patient_df.select(
        "patient_id",
        "first_name",
        "last_name",
        "gender",
        "birth_date",
        "active"
    )
)

In [0]:
silver_patient_df = (
    silver_patient_df
    .withColumn(
        "city",
        get_json_object(
            col("raw_json"),
            "$.address[0].city"
        )
    )
    .withColumn(
        "state",
        get_json_object(
            col("raw_json"),
            "$.address[0].state"
        )
    )
    .withColumn(
        "postal_code",
        get_json_object(
            col("raw_json"),
            "$.address[0].postalCode"
        )
    )
)

display(
    silver_patient_df.select(
        "patient_id",
        "first_name",
        "last_name",
        "gender",
        "birth_date",
        "city",
        "state",
        "postal_code"
    )
)

In [0]:
silver_patient_df = (
    silver_patient_df
    .withColumn(
        "first_name",
        trim(col("first_name"))
    )
    .withColumn(
        "last_name",
        trim(col("last_name"))
    )
    .withColumn(
        "gender",
        lower(trim(col("gender")))
    )
    .withColumn(
        "city",
        trim(col("city"))
    )
    .withColumn(
        "state",
        trim(col("state"))
    )
    .withColumn(
        "postal_code",
        trim(col("postal_code"))
    )
)

display(silver_patient_df)

In [0]:
silver_patient_df = (
    silver_patient_df
    .withColumn(
        "data_quality_status",
        when(
            col("patient_id").isNull(),
            lit("INVALID")
        ).otherwise(
            lit("VALID")
        )
    )
)

display(
    silver_patient_df.select(
        "patient_id",
        "first_name",
        "last_name",
        "data_quality_status"
    )
)

In [0]:
display(
    silver_patient_df
    .filter(
        col("data_quality_status") == "INVALID"
    )
)

In [0]:
invalid_count = (
    silver_patient_df
    .filter(
        col("data_quality_status") == "INVALID"
    )
    .count()
)

print("Invalid Patient records:", invalid_count)

In [0]:
patient_window = (
    Window
    .partitionBy("patient_id")
    .orderBy(
        col("ingestion_timestamp").desc()
    )
)

In [0]:
silver_patient_df = (
    silver_patient_df
    .withColumn(
        "row_number",
        row_number().over(patient_window)
    )
)

In [0]:
display(
    silver_patient_df.select(
        "patient_id",
        "ingestion_timestamp",
        "row_number"
    )
)

In [0]:
silver_patient_df = (
    silver_patient_df
    .filter(
        col("row_number") == 1
    )
    .drop("row_number")
)

print(
    "Records after deduplication:",
    silver_patient_df.count()
)

In [0]:
silver_patient_df = (
    silver_patient_df
    .withColumn(
        "record_hash",
        sha2(
            concat_ws(
                "||",
                col("patient_id"),
                col("first_name"),
                col("last_name"),
                col("gender"),
                col("birth_date"),
                col("city"),
                col("state"),
                col("postal_code"),
                col("active")
            ),
            256
        )
    )
)

display(
    silver_patient_df.select(
        "patient_id",
        "record_hash"
    )
)

In [0]:
silver_patient_df = (
    silver_patient_df
    .withColumn(
        "effective_start_date",
        to_date(col("ingestion_timestamp"))
    )
    .withColumn(
        "effective_end_date",
        lit(None).cast("date")
    )
    .withColumn(
        "is_current",
        lit(True)
    )
)

In [0]:
silver_patient_df = (
    silver_patient_df
    .select(
        "patient_id",
        "first_name",
        "last_name",
        "gender",
        "birth_date",
        "active",
        "city",
        "state",
        "postal_code",
        "data_quality_status",
        "record_hash",
        "effective_start_date",
        "effective_end_date",
        "is_current",
        "source_file",
        "ingestion_timestamp",
        "raw_json"
    )
)

display(silver_patient_df)

In [0]:
print(
    "Final Silver Patient count:",
    silver_patient_df.count()
)

In [0]:
display(
    silver_patient_df.select(
        "patient_id",
        "first_name",
        "last_name",
        "gender",
        "birth_date",
        "city",
        "state",
        "record_hash",
        "effective_start_date",
        "effective_end_date",
        "is_current"
    ).limit(10)
)

In [0]:
(
    silver_patient_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_TABLE)
)

print(
    "Silver Patient table created successfully."
)

In [0]:
display(
    spark.table("workspace.fhir.silver_patient")
)

In [0]:
print(
    "Silver Patient count:",
    spark.table(
        "workspace.fhir.silver_patient"
    ).count()
)

In [0]:
display(
    spark.table("workspace.fhir.silver_patient")
    .select(
        "patient_id",
        "record_hash",
        "effective_start_date",
        "effective_end_date",
        "is_current"
    )
    .limit(10)
)

In [0]:
display(
    spark.table("workspace.fhir.silver_patient")
    .select(
        "patient_id",
        "first_name",
        "last_name",
        "city",
        "state",
        "record_hash",
        "effective_start_date",
        "effective_end_date",
        "is_current"
    )
    .limit(20)
)

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.fhir.silver_patient_scd2
USING DELTA
AS
SELECT
    patient_id,
    first_name,
    last_name,
    gender,
    birth_date,
    active,
    city,
    state,
    postal_code,
    data_quality_status,
    record_hash,
    effective_start_date,
    effective_end_date,
    is_current,
    source_file,
    ingestion_timestamp,
    raw_json
FROM workspace.fhir.silver_patient
""")

In [0]:
display(
    spark.table("workspace.fhir.silver_patient_scd2")
)

In [0]:
print(
    "SCD2 records:",
    spark.table(
        "workspace.fhir.silver_patient_scd2"
    ).count()
)

In [0]:
display(
    spark.table("workspace.fhir.silver_patient_scd2")
    .select(
        "patient_id",
        "city",
        "state",
        "effective_start_date",
        "effective_end_date",
        "is_current"
    )
    .limit(10)
)

In [0]:
current_patient_df = spark.table(
    "workspace.fhir.silver_patient"
)

In [0]:
print(
    "SCD2 records:",
    spark.table(
        "workspace.fhir.silver_patient_scd2"
    ).count()
)

In [0]:
display(
    spark.table("workspace.fhir.silver_patient_scd2")
    .select(
        "patient_id",
        "city",
        "state",
        "effective_start_date",
        "effective_end_date",
        "is_current"
    )
    .limit(10)
)

In [0]:
current_patient_df = spark.table(
    "workspace.fhir.silver_patient"
)

In [0]:
display(
    current_patient_df.limit(5)
)

In [0]:
from delta.tables import DeltaTable

silver_patient_scd2_table = DeltaTable.forName(
    spark,
    "workspace.fhir.silver_patient_scd2"
)

print("Delta table loaded successfully.")

In [0]:
from pyspark.sql.functions import current_date

(
    silver_patient_scd2_table.alias("target")
    .merge(
        current_patient_df.alias("source"),
        """
        target.patient_id = source.patient_id
        AND target.is_current = true
        AND target.record_hash <> source.record_hash
        """
    )
    .whenMatchedUpdate(
        set={
            "effective_end_date": current_date(),
            "is_current": lit(False)
        }
    )
    .execute()
)

print("Existing changed records closed.")

In [0]:
(
    silver_patient_scd2_table.alias("target")
    .merge(
        current_patient_df.alias("source"),
        """
        target.patient_id = source.patient_id
        AND target.is_current = true
        """
    )
    .whenNotMatchedInsert(
        values={
            "patient_id": "source.patient_id",
            "first_name": "source.first_name",
            "last_name": "source.last_name",
            "gender": "source.gender",
            "birth_date": "source.birth_date",
            "active": "source.active",
            "city": "source.city",
            "state": "source.state",
            "postal_code": "source.postal_code",
            "data_quality_status": "source.data_quality_status",
            "record_hash": "source.record_hash",
            "effective_start_date": "source.effective_start_date",
            "effective_end_date": "source.effective_end_date",
            "is_current": "source.is_current",
            "source_file": "source.source_file",
            "ingestion_timestamp": "source.ingestion_timestamp",
            "raw_json": "source.raw_json"
        }
    )
    .execute()
)

print("New records / new versions inserted.")

In [0]:
display(
    spark.table("workspace.fhir.silver_patient_scd2")
    .select(
        "patient_id",
        "city",
        "state",
        "record_hash",
        "effective_start_date",
        "effective_end_date",
        "is_current"
    )
    .orderBy("patient_id")
)

In [0]:
BRONZE_TABLE = "workspace.fhir.bronze_encounter"
SILVER_TABLE = "workspace.fhir.silver_encounter"

bronze_encounter_df = spark.table(BRONZE_TABLE)

print(
    "Bronze Encounter count:",
    bronze_encounter_df.count()
)

In [0]:
silver_encounter_df = (
    bronze_encounter_df
    .select(
        col("resource_id").alias("encounter_id"),

        get_json_object(
            col("raw_json"),
            "$.status"
        ).alias("status"),

        get_json_object(
            col("raw_json"),
            "$.class.code"
        ).alias("class_code"),

        get_json_object(
            col("raw_json"),
            "$.subject.reference"
        ).alias("patient_reference"),

        get_json_object(
            col("raw_json"),
            "$.period.start"
        ).alias("period_start"),

        get_json_object(
            col("raw_json"),
            "$.period.end"
        ).alias("period_end"),

        col("source_file"),
        col("ingestion_timestamp"),
        col("raw_json")
    )
)

display(silver_encounter_df)

In [0]:
silver_encounter_df = (
    silver_encounter_df
    .withColumn(
        "status",
        lower(trim(col("status")))
    )
    .withColumn(
        "class_code",
        trim(col("class_code"))
    )
    .withColumn(
        "patient_reference",
        trim(col("patient_reference"))
    )
)

In [0]:
silver_encounter_df = (
    silver_encounter_df
    .withColumn(
        "data_quality_status",
        when(
            col("encounter_id").isNull(),
            lit("INVALID")
        ).otherwise(
            lit("VALID")
        )
    )
)

In [0]:
encounter_window = (
    Window
    .partitionBy("encounter_id")
    .orderBy(
        col("ingestion_timestamp").desc()
    )
)

silver_encounter_df = (
    silver_encounter_df
    .withColumn(
        "row_number",
        row_number().over(encounter_window)
    )
    .filter(
        col("row_number") == 1
    )
    .drop("row_number")
)

In [0]:
silver_encounter_df = (
    silver_encounter_df
    .withColumn(
        "record_hash",
        sha2(
            concat_ws(
                "||",
                col("encounter_id"),
                col("status"),
                col("class_code"),
                col("patient_reference"),
                col("period_start"),
                col("period_end")
            ),
            256
        )
    )
)

In [0]:
silver_encounter_df = (
    silver_encounter_df
    .withColumn(
        "effective_start_date",
        to_date(col("ingestion_timestamp"))
    )
    .withColumn(
        "effective_end_date",
        lit(None).cast("date")
    )
    .withColumn(
        "is_current",
        lit(True)
    )
)

In [0]:
(
    silver_encounter_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.fhir.silver_encounter"
    )
)

print("Silver Encounter created successfully.")

In [0]:
display(
    spark.table("workspace.fhir.silver_encounter")
)

In [0]:
display(
    spark.table("workspace.fhir.silver_encounter")
)

In [0]:
bronze_observation_df = spark.table(
    "workspace.fhir.bronze_observation"
)

print(
    "Bronze Observation count:",
    bronze_observation_df.count()
)

In [0]:
silver_observation_df = (
    bronze_observation_df
    .select(
        col("resource_id").alias("observation_id"),

        get_json_object(
            col("raw_json"),
            "$.status"
        ).alias("status"),

        get_json_object(
            col("raw_json"),
            "$.code.coding[0].code"
        ).alias("observation_code"),

        get_json_object(
            col("raw_json"),
            "$.code.coding[0].display"
        ).alias("observation_name"),

        get_json_object(
            col("raw_json"),
            "$.subject.reference"
        ).alias("patient_reference"),

        get_json_object(
            col("raw_json"),
            "$.valueQuantity.value"
        ).alias("value"),

        get_json_object(
            col("raw_json"),
            "$.valueQuantity.unit"
        ).alias("unit"),

        col("source_file"),
        col("ingestion_timestamp"),
        col("raw_json")
    )
)

display(silver_observation_df)

In [0]:
silver_observation_df = (
    silver_observation_df
    .withColumn(
        "status",
        lower(trim(col("status")))
    )
    .withColumn(
        "observation_code",
        trim(col("observation_code"))
    )
    .withColumn(
        "patient_reference",
        trim(col("patient_reference"))
    )
    .withColumn(
        "data_quality_status",
        when(
            col("observation_id").isNull(),
            lit("INVALID")
        ).otherwise(
            lit("VALID")
        )
    )
)

In [0]:
observation_window = (
    Window
    .partitionBy("observation_id")
    .orderBy(
        col("ingestion_timestamp").desc()
    )
)

silver_observation_df = (
    silver_observation_df
    .withColumn(
        "row_number",
        row_number().over(observation_window)
    )
    .filter(
        col("row_number") == 1
    )
    .drop("row_number")
)

In [0]:
silver_observation_df = (
    silver_observation_df
    .withColumn(
        "record_hash",
        sha2(
            concat_ws(
                "||",
                col("observation_id"),
                col("status"),
                col("observation_code"),
                col("observation_name"),
                col("patient_reference"),
                col("value"),
                col("unit")
            ),
            256
        )
    )
    .withColumn(
        "effective_start_date",
        to_date(col("ingestion_timestamp"))
    )
    .withColumn(
        "effective_end_date",
        lit(None).cast("date")
    )
    .withColumn(
        "is_current",
        lit(True)
    )
)

In [0]:
(
    silver_observation_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.fhir.silver_observation"
    )
)

print("Silver Observation created successfully.")

In [0]:
display(
    spark.table("workspace.fhir.silver_observation")
)

In [0]:
bronze_condition_df = spark.table(
    "workspace.fhir.bronze_condition"
)

print(
    "Bronze Condition count:",
    bronze_condition_df.count()
)

In [0]:
silver_condition_df = (
    bronze_condition_df
    .select(
        col("resource_id").alias("condition_id"),

        get_json_object(
            col("raw_json"),
            "$.clinicalStatus.coding[0].code"
        ).alias("clinical_status"),

        get_json_object(
            col("raw_json"),
            "$.verificationStatus.coding[0].code"
        ).alias("verification_status"),

        get_json_object(
            col("raw_json"),
            "$.code.coding[0].code"
        ).alias("condition_code"),

        get_json_object(
            col("raw_json"),
            "$.code.coding[0].display"
        ).alias("condition_name"),

        get_json_object(
            col("raw_json"),
            "$.subject.reference"
        ).alias("patient_reference"),

        col("source_file"),
        col("ingestion_timestamp"),
        col("raw_json")
    )
)

display(silver_condition_df)

In [0]:
silver_condition_df = (
    silver_condition_df
    .withColumn(
        "clinical_status",
        lower(trim(col("clinical_status")))
    )
    .withColumn(
        "verification_status",
        lower(trim(col("verification_status")))
    )
    .withColumn(
        "condition_code",
        trim(col("condition_code"))
    )
    .withColumn(
        "patient_reference",
        trim(col("patient_reference"))
    )
    .withColumn(
        "data_quality_status",
        when(
            col("condition_id").isNull(),
            lit("INVALID")
        ).otherwise(
            lit("VALID")
        )
    )
)

In [0]:
condition_window = (
    Window
    .partitionBy("condition_id")
    .orderBy(
        col("ingestion_timestamp").desc()
    )
)

silver_condition_df = (
    silver_condition_df
    .withColumn(
        "row_number",
        row_number().over(condition_window)
    )
    .filter(
        col("row_number") == 1
    )
    .drop("row_number")
)

In [0]:
silver_condition_df = (
    silver_condition_df
    .withColumn(
        "record_hash",
        sha2(
            concat_ws(
                "||",
                col("condition_id"),
                col("clinical_status"),
                col("verification_status"),
                col("condition_code"),
                col("condition_name"),
                col("patient_reference")
            ),
            256
        )
    )
    .withColumn(
        "effective_start_date",
        to_date(col("ingestion_timestamp"))
    )
    .withColumn(
        "effective_end_date",
        lit(None).cast("date")
    )
    .withColumn(
        "is_current",
        lit(True)
    )
)

In [0]:
(
    silver_condition_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.fhir.silver_condition"
    )
)

print("Silver Condition created successfully.")

In [0]:
display(
    spark.table("workspace.fhir.silver_condition")
)

In [0]:
silver_tables = [
    "silver_patient",
    "silver_encounter",
    "silver_observation",
    "silver_condition"
]

for table in silver_tables:

    table_name = f"workspace.fhir.{table}"

    df = spark.table(table_name)

    print(
        f"{table_name}: {df.count()} records"
    )

In [0]:
for table in silver_tables:

    table_name = f"workspace.fhir.{table}"

    print("================================")
    print(table_name)

    display(
        spark.table(table_name)
        .groupBy("data_quality_status")
        .count()
    )

In [0]:
display(
    spark.table("workspace.fhir.silver_patient")
    .select(
        "patient_id",
        "record_hash",
        "effective_start_date",
        "effective_end_date",
        "is_current"
    )
    .limit(10)
)

In [0]:
display(
    spark.table("workspace.fhir.silver_encounter")
    .select(
        "encounter_id",
        "record_hash",
        "effective_start_date",
        "effective_end_date",
        "is_current"
    )
    .limit(10)
)

In [0]:
display(
    spark.table("workspace.fhir.silver_observation")
    .select(
        "observation_id",
        "record_hash",
        "effective_start_date",
        "effective_end_date",
        "is_current"
    )
    .limit(10)
)

In [0]:
display(
    spark.table("workspace.fhir.silver_condition")
    .select(
        "condition_id",
        "record_hash",
        "effective_start_date",
        "effective_end_date",
        "is_current"
    )
    .limit(10)
)